In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report


try:
    df = pd.read_csv("aviation-accident.csv", sep=";", encoding="latin1")
except:
    df = pd.read_csv("aviation-accident.csv", sep=";", encoding="cp1252")

df.columns = df.columns.str.strip().str.lower()
df["fatalities"] = pd.to_numeric(df["fatalities"], errors="coerce").fillna(0)
df["year"] = pd.to_numeric(df["year"], errors="coerce").fillna(df["year"].median())

df["fatal_accident"] = (df["fatalities"] > 0).astype(int)

features = ["country", "operator", "type", "cat", "year"]
X = df[features].fillna("Unknown")
y = df["fatal_accident"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("train:", X_train.shape)
print("test:", X_test.shape)

train: (19084, 5)
test: (4772, 5)


In [13]:
categorical_features = features

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)

print("Modell trainiert.")
#Für diesen Datensatz eignet sich der Random Forest Classifier besonders gut,
#da Accidents von vielen verschiedenen Faktoren wie Flugzeugtyp, Betreiber,
#Land, Jahr oder Unfallkategorie beeinflusst werden. Dieses Modell kann komplexe
#Zusammenhänge zwischen mehreren Variablen erkennen und verarbeitet auch grosse
#Datenmengen hut. Durch die Kombination vieler Entscheidungsbäume
#liefert Random Forest stabilere und genauere Resultate als ein einzelnerEntscheidungsbaum. 

Modell trainiert.


In [14]:
y_pred = model.predict(X_test)

results = X_test.copy()
results['Tatsächlich'] = y_test
results['Vorhersage'] = y_pred

print(results.head(10))
#Das Modell erreicht eine Accuracy von rund 70 %, was bedeutet, 
#dass etwa sieben von zehn Unfällen korrekt als tödlich oder nicht tödlich klassifiziert werden. 
#Besonders bei nicht tödlichen Unfällen arbeitet das Modell zuverlässiger als bei tödlichen. 
#Insgesamt liefert das Modell ok-brauchbare Resultate, jedoch entstehen Fehler bei komplexeren oder ungewöhnlicheren Fällen. 

             country          operator                         type cat  year  \
2862   Pacific Ocean           US Navy  Consolidated PBY-5 Catalina  A1  1944   
13167      Nicaragua     Nicaraguan AF           U-1A Otter (DHC-3)  A1  1976   
15722      Indonesia            Garuda                      DC-9-32  A1  1987   
21420         Canada    Wasaya Airways                  Beech 1900D  A2  2011   
19507           U.K.             Flybe                  BAe-146-200  A2  2002   
21399         Russia              VASO          Antonov An-148-100E  A1  2011   
1713           Sudan  South African AF         Lockheed 18 Lodestar  A1  1943   
8729           Libya           US Army           U-1A Otter (DHC-3)  A1  1960   
640      Netherlands         German AF             Junkers Ju-52/3m  A1  1940   
10202        Vietnam       Air America         Douglas C-47A (DC-3)  C1  1966   

       Tatsächlich  Vorhersage  
2862             0           1  
13167            1           1  
15722    